# BME i9400 — Meeting 2
## Probability for Diagnosis

**Fall 2026 · Wednesday, September 2**

Today is the only lecture in this course that is purely about probability. Everything in it comes
back later: the table we build in the first ten minutes is the *confusion matrix* we will use to
evaluate every classifier from Meeting 9 onward.

**Where we are going**

1. The four outcomes of a test
2. Conditional probability and the law of total probability
3. Bayes' rule
4. A real screening problem, solved two ways
5. Sensitivity, specificity, PPV, NPV — and which of them depend on the population
6. Likelihood ratios
7. Why this is the whole course in miniature

---
## The problem

Screening mammography in the United States performs roughly as follows
(Breast Cancer Surveillance Consortium, 1.68 million screening mammograms, 2007–2013):

- **Sensitivity 86.9%** — of women who have breast cancer, 86.9% are flagged by the mammogram
- **Specificity 88.9%** — of women who do not, 88.9% are correctly called negative
- About **6 in 1,000** women screened have breast cancer

> ### A woman in a routine screening program receives a positive mammogram.
> ### What is the probability that she has breast cancer?

**Commit to an answer before we go on.** Roughly 90%? About 50%? Around 10%? Under 5%?

Write it down. We will come back to it.

---
## Step 1 — The four outcomes

Two binary variables. That is the entire setup.

- $D$ — disease status: $D=1$ (has the disease), $D=0$ (does not)
- $T$ — test result: $T=1$ (positive), $T=0$ (negative)

Two binary variables give four possible combinations, and every one has a name:

| | $T=1$ (test positive) | $T=0$ (test negative) |
|---|---|---|
| **$D=1$ (diseased)** | True Positive (TP) | False Negative (FN) |
| **$D=0$ (healthy)** | False Positive (FP) | True Negative (TN) |

Look at this table carefully. In Meeting 9 we will call it a **confusion matrix** and use it to
evaluate machine learning classifiers. It is the same four boxes. The only thing that changes is
that $T$ becomes the output of a model instead of the output of a machine in radiology.

Notice that the two kinds of error are *not* interchangeable. A false negative sends a woman with
cancer home. A false positive sends a healthy woman for a biopsy. Which is worse depends entirely on
the clinical context — and choosing between them is a decision, not a calculation.

---
## Step 2 — Conditional probability

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

Read it as: *among the cases where $B$ happened, what fraction also had $A$?* The conditioning event
$B$ becomes the new denominator — we have restricted attention to a sub-population.

This is the whole idea, and it is where the intuition usually breaks: **$P(A \mid B)$ and
$P(B \mid A)$ are different quantities.** They have different denominators.

In our problem:

$$P(T=1 \mid D=1) = 0.869 \quad \text{is the sensitivity — among diseased women, the fraction flagged}$$

$$P(D=1 \mid T=1) = \;? \quad \text{is what the patient wants to know — among flagged women, the fraction diseased}$$

The first is a property of the test, measured in a study. The second is what you actually care about
in the clinic. Confusing them is called the **prosecutor's fallacy**, and it is the single most
common probabilistic error in medicine.

### The law of total probability

To get from one to the other we need $P(T=1)$: how often the test is positive *at all*. A woman
either has the disease or she does not, so we can split the population in two and add up:

$$P(T=1) = \underbrace{P(T=1 \mid D=1)\,P(D=1)}_{\text{positives from diseased women}} \;+\; \underbrace{P(T=1 \mid D=0)\,P(D=0)}_{\text{positives from healthy women}}$$

More generally, for any partition $\{B_i\}$ of the sample space: $P(A) = \sum_i P(A \mid B_i) P(B_i)$.

---
## Step 3 — Bayes' rule

Start from the fact that $P(A \cap B)$ and $P(B \cap A)$ are the same thing:

$$P(A \mid B)\,P(B) = P(A \cap B) = P(B \mid A)\,P(A)$$

Divide through by $P(B)$:

$$\boxed{\;P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)}\;}$$

Expanding the denominator with the law of total probability gives the form we will use:

$$P(D=1 \mid T=1) = \frac{P(T=1 \mid D=1)\,P(D=1)}{P(T=1 \mid D=1)\,P(D=1) + P(T=1 \mid D=0)\,P(D=0)}$$

$$\text{posterior} = \frac{\text{likelihood} \times \text{prior}}{\text{evidence}}$$

Bayes' rule is a machine for **reversing a conditional**. You know the test's behaviour given the
disease; you want the disease given the test. That reversal costs you one piece of information: the
prior, $P(D=1)$ — the prevalence. Everything surprising about today comes from that term.

---
## Step 4a — Solve it with algebra

$$P(D{=}1) = 0.006 \qquad P(T{=}1 \mid D{=}1) = 0.869 \qquad P(T{=}1 \mid D{=}0) = 1 - 0.889 = 0.111$$

**Evidence:**

$$P(T{=}1) = (0.869)(0.006) + (0.111)(0.994) = 0.00521 + 0.11033 = 0.1155$$

**Posterior:**

$$P(D{=}1 \mid T{=}1) = \frac{0.00521}{0.1155} = 0.045$$

### About 4.5%.

Not 90%. A woman with a positive screening mammogram has roughly a **1 in 22** chance of having
breast cancer.

In [ ]:
# Verify the arithmetic, and set up the numbers we will reuse all lecture.
SENS = 0.869      # P(T=1 | D=1)   BCSC national benchmark
SPEC = 0.889      # P(T=0 | D=0)
PREV = 0.006      # P(D=1)         about 6 per 1000 women screened

def ppv(prev, sens=SENS, spec=SPEC):
    """P(disease | positive test)."""
    return sens * prev / (sens * prev + (1 - spec) * (1 - prev))

def npv(prev, sens=SENS, spec=SPEC):
    """P(no disease | negative test)."""
    return spec * (1 - prev) / (spec * (1 - prev) + (1 - sens) * prev)

evidence = SENS * PREV + (1 - SPEC) * (1 - PREV)
print(f"P(T=1)              = {evidence:.4f}    (the test is positive {100*evidence:.1f}% of the time)")
print(f"P(D=1 | T=1)  PPV   = {ppv(PREV):.4f}    ->  {100*ppv(PREV):.1f}%")
print(f"P(D=0 | T=0)  NPV   = {npv(PREV):.4f}    ->  {100*npv(PREV):.2f}%")

---
## Step 4b — Solve it again, by counting people

The algebra is correct but it does not *feel* like anything. Here is the same calculation with no
formulas at all. Imagine **1,000 women** walking into a screening program.

- **6 have breast cancer.** The test catches 86.9% of them → **5 test positive**, 1 is missed.
- **994 do not.** The test wrongly flags 11.1% of them → **110 test positive**, 884 are cleared.

So **115 women get a positive result**, and only **5 of them have cancer**.

$$\frac{5}{115} = 4.3\%$$

That is it. That is Bayes' rule. The reason the answer is small is not subtle: **the healthy group is
so much larger that even a small error rate on it produces more false positives than there are true
cases.** 11% of 994 simply beats 87% of 6.

Whenever a Bayes problem feels counterintuitive, count people instead of multiplying probabilities.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 13, "figure.dpi": 110})

# ---- Figure 1: 1,000 women, one square each -------------------------------
N = 1000
n_disease = round(N * PREV)                      # 6
n_tp = round(n_disease * SENS)                   # 5
n_fn = n_disease - n_tp                          # 1
n_fp = round((N - n_disease) * (1 - SPEC))       # 110
n_tn = N - n_disease - n_fp                      # 884

COLORS = {"TN": "#dcd8e3", "FP": "#e8a33d", "FN": "#8c8898", "TP": "#c1272d"}
cells = ["TN"] * n_tn + ["FP"] * n_fp + ["FN"] * n_fn + ["TP"] * n_tp

rows, cols = 25, 40
fig, ax = plt.subplots(figsize=(13, 8.5))
for i, kind in enumerate(cells):
    r, c = divmod(i, cols)
    ax.add_patch(plt.Rectangle((c, rows - 1 - r), 0.86, 0.86,
                               facecolor=COLORS[kind], edgecolor="none"))
ax.set_xlim(-0.5, cols); ax.set_ylim(-0.5, rows); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("1,000 women screened", fontsize=19, weight="bold", pad=16)

handles = [plt.Rectangle((0, 0), 1, 1, facecolor=COLORS[k]) for k in ["TN", "FP", "FN", "TP"]]
labels  = [f"True negative — cleared, healthy   ({n_tn})",
           f"FALSE POSITIVE — healthy, flagged   ({n_fp})",
           f"False negative — cancer, missed   ({n_fn})",
           f"TRUE POSITIVE — cancer, caught   ({n_tp})"]
ax.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, -0.02),
          ncol=2, frameon=False, fontsize=13)
plt.tight_layout(); plt.show()

print(f"Positive results:  {n_tp + n_fp}      of which real cancers: {n_tp}")
print(f"PPV = {n_tp}/{n_tp + n_fp} = {n_tp/(n_tp+n_fp):.3f}  ->  {100*n_tp/(n_tp+n_fp):.1f}%")

### This is not a thought experiment

The BCSC study reports an **observed PPV of 4.4%** across 1.68 million screening mammograms — meaning
that of all the women called back from a positive screen, 4.4% turned out to have cancer.

Our calculation said 4.5%. The same arithmetic also predicted that the test would come back
positive 11.6% of the time — and BCSC reports an abnormal interpretation rate of **11.6%**. The arithmetic on this slide predicts what a decade of national screening
data actually shows. This is worth pausing on: probability applied carefully to two published numbers
told us something true about the world before we looked.

*Source: Lehman et al., "National Performance Benchmarks for Modern Screening Digital Mammography:
Update from the Breast Cancer Surveillance Consortium," Radiology, 2017.*

---
## Step 5 — The vocabulary, and which parts move

Four quantities, two of which are constantly confused:

| Term | Definition | Reads across | Depends on prevalence? |
|---|---|---|---|
| **Sensitivity** (TPR, recall) | $P(T{=}1 \mid D{=}1)$ | the diseased row | **No** |
| **Specificity** (TNR) | $P(T{=}0 \mid D{=}0)$ | the healthy row | **No** |
| **PPV** (precision) | $P(D{=}1 \mid T{=}1)$ | the positive column | **Yes** |
| **NPV** | $P(D{=}0 \mid T{=}0)$ | the negative column | **Yes** |

> **The distinction to hold onto.** Sensitivity and specificity are properties of **the test**. PPV and
> NPV are properties of **the test applied to a particular population**. Take the identical machine
> and the identical radiologist to a different clinic, and the sensitivity is unchanged while the PPV
> may move by an order of magnitude.

Sensitivity and specificity condition on disease status, so they are computed *within* the diseased
or healthy group and never see the group sizes. PPV and NPV condition on the test result, so both
groups are mixed in the denominator — and the mixture is set by prevalence.

*(A common slip: specificity is not "the probability that a positive test is correct." That is PPV.
Specificity is about the healthy, and it is about negative results.)*

In [ ]:
# ---- Figure 2: PPV and NPV as prevalence changes --------------------------
prev_grid = np.logspace(-4, -0.3, 400)          # 0.01% to 50%

fig, ax = plt.subplots(figsize=(12, 6.5))
ax.semilogx(prev_grid, 100 * ppv(prev_grid), lw=3, color="#c1272d", label="PPV — $P(D{=}1 \\mid T{=}1)$")
ax.semilogx(prev_grid, 100 * npv(prev_grid), lw=3, color="#2b6cb0", label="NPV — $P(D{=}0 \\mid T{=}0)$")
ax.axhline(100 * SENS, ls="--", lw=2, color="#6e6878")
ax.text(1.2e-4, 100 * SENS + 2.5, "sensitivity (86.9%) — flat, does not depend on prevalence",
        fontsize=12, color="#6e6878")

# (dy, x-multiplier) keep the three labels from colliding with each other
# and with the sensitivity line
settings = [(0.006, "Screening\n(6 per 1,000)",         +18, 1.00),
            (0.05,  "Symptomatic clinic\n(5%)",         +20, 0.28),
            (0.25,  "High-risk / palpable mass\n(25%)", -30, 1.05)]
for p, label, dy, xmul in settings:
    ax.plot(p, 100 * ppv(p), "o", ms=13, color="#c1272d", zorder=5)
    ax.annotate(f"{label}\nPPV = {100*ppv(p):.1f}%", xy=(p, 100 * ppv(p)),
                xytext=(p * xmul, 100 * ppv(p) + dy), ha="center", fontsize=12,
                arrowprops=dict(arrowstyle="-", color="#c1272d", lw=1.4))

ax.set_xlabel("prevalence  $P(D{=}1)$  — log scale", fontsize=15)
ax.set_ylabel("percent", fontsize=15)
ax.set_title("The same test in three different populations", fontsize=18, weight="bold", pad=14)
ax.set_ylim(0, 108); ax.legend(loc="center left", fontsize=13, framealpha=0.95)
ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

### Read that figure carefully

One test. Three clinics. The PPV runs from about 4% to about 72%, and **nothing about the test
changed** — only who walked through the door.

This is why "our model achieves 94% precision" is an incomplete claim. Precision *is* PPV, and PPV is
a statement about a population as much as about a model. When we get to Meeting 9 and start reporting
precision on imbalanced clinical datasets, this figure is the reason we will insist on knowing the
prevalence of the test set.

It is also why a model trained at one hospital can post excellent numbers and then disappoint at
another. Nothing broke. The prevalence moved.

---
## Step 6 — Likelihood ratios: the practical shortcut

Rewriting Bayes in **odds** form removes the awkward denominator entirely.

Odds and probability are two ways of saying the same thing:
$\text{odds} = \frac{p}{1-p}$, and $p = \frac{\text{odds}}{1+\text{odds}}$.

$$\boxed{\;\text{posterior odds} = \text{LR} \times \text{prior odds}\;}$$

where the **likelihood ratio** for a positive test is

$$\text{LR}^{+} = \frac{P(T{=}1 \mid D{=}1)}{P(T{=}1 \mid D{=}0)} = \frac{\text{sensitivity}}{1 - \text{specificity}}
= \frac{0.869}{0.111} = 7.8$$

and for a negative test

$$\text{LR}^{-} = \frac{1 - \text{sensitivity}}{\text{specificity}} = \frac{0.131}{0.889} = 0.147$$

**Why this is the useful form.** The likelihood ratio is a property of the test alone, and it says
exactly one thing: *how much does this result move my belief?* A positive mammogram multiplies your
odds of cancer by about 8. If you started at long odds, 8× still leaves you at long odds — which is
the entire mammography result in one sentence.

Rules of thumb clinicians actually use: LR > 10 is strong evidence, 5–10 moderate, 2–5 weak,
and near 1 means the test told you nothing.

In [ ]:
# ---- The same patient, worked in odds ------------------------------------
lr_pos = SENS / (1 - SPEC)
lr_neg = (1 - SENS) / SPEC

prior_odds     = PREV / (1 - PREV)
posterior_odds = lr_pos * prior_odds
posterior_p    = posterior_odds / (1 + posterior_odds)

print(f"LR+  = {SENS:.3f} / {1-SPEC:.3f} = {lr_pos:.2f}")
print(f"LR-  = {1-SENS:.3f} / {SPEC:.3f} = {lr_neg:.3f}\n")
print(f"prior odds      = {PREV:.4f} / {1-PREV:.4f} = {prior_odds:.5f}   (about 1 : {1/prior_odds:.0f})")
print(f"posterior odds  = {lr_pos:.2f} x {prior_odds:.5f} = {posterior_odds:.4f}   (about 1 : {1/posterior_odds:.0f})")
print(f"posterior prob  = {posterior_p:.4f}  ->  {100*posterior_p:.1f}%   (matches Step 4)\n")

print("How far one positive mammogram moves you, from different starting points:")
print(f"{'prior':>10} {'prior odds':>12} {'posterior odds':>16} {'posterior':>11}")
for p in [0.006, 0.05, 0.25, 0.50]:
    po = lr_pos * (p / (1 - p))
    print(f"{p:>9.1%} {p/(1-p):>12.4f} {po:>16.3f} {po/(1+po):>10.1%}")

---
## Step 7 — Why this is the whole course in miniature

Three ideas from today reappear in almost every remaining meeting.

**1. The four boxes become the confusion matrix.** Replace "the mammogram was positive" with "the
model predicted class 1" and nothing else changes. Meeting 9 is built on this table.

**2. A test has a knob, and turning it trades one error for the other.** A radiologist reading more
aggressively catches more cancers *and* generates more false alarms. A classifier has exactly the
same knob — the threshold on its predicted probability — and sweeping it traces out the ROC curve we
will meet in Meeting 9. Sensitivity and specificity are not fixed properties handed down from above;
they are a point you *choose*.

**3. Prevalence is not the model's business, but it is yours.** Sensitivity and specificity survive a
change of population. PPV does not. Any performance claim that does not tell you the prevalence of
the evaluation set is incomplete — a point we will return to when you report results in your capstone.

The figure below is the preview: a model outputs a score, the two groups overlap, and *you* pick
where to cut.

In [ ]:
# ---- Figure 3: the threshold is a choice ---------------------------------
from scipy.stats import norm

x = np.linspace(-4, 8, 600)
healthy, diseased = norm(1.0, 1.0), norm(4.0, 1.2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
for ax, thr, title in zip(axes, [1.6, 2.5, 3.2],
                          ["Aggressive threshold", "Balanced", "Conservative threshold"]):
    ax.fill_between(x, healthy.pdf(x), color="#2b6cb0", alpha=0.30)
    ax.fill_between(x, diseased.pdf(x), color="#c1272d", alpha=0.30)
    ax.axvline(thr, color="k", lw=2.5)
    sens_t = 1 - diseased.cdf(thr)
    spec_t = healthy.cdf(thr)
    ax.set_title(f"{title}\nsens {sens_t:.0%} · spec {spec_t:.0%}", fontsize=13)
    ax.set_xlabel("model score"); ax.set_yticks([])
    ax.annotate("call positive →", xy=(thr + 0.15, 0.36), fontsize=10.5)

axes[0].set_ylabel("density")
fig.legend([plt.Rectangle((0,0),1,1,facecolor="#2b6cb0",alpha=0.3),
            plt.Rectangle((0,0),1,1,facecolor="#c1272d",alpha=0.3)],
           ["healthy", "diseased"], loc="upper right", frameon=False, fontsize=12)
fig.suptitle("Same model, same data — three different operating points",
             fontsize=17, weight="bold", y=1.04)
plt.tight_layout(); plt.show()

---
## What to take away

**Definitions you must know cold** — these are examinable and they recur all semester:

- **Sensitivity** $= P(T{=}1 \mid D{=}1)$ — among the diseased, the fraction detected
- **Specificity** $= P(T{=}0 \mid D{=}0)$ — among the healthy, the fraction cleared
- **PPV** $= P(D{=}1 \mid T{=}1)$ — among positives, the fraction truly diseased
- **NPV** $= P(D{=}0 \mid T{=}0)$ — among negatives, the fraction truly healthy
- **Prevalence** $= P(D{=}1)$ — the prior
- **Bayes' rule** — the machine for reversing a conditional
- **LR$^+$** $= \dfrac{\text{sens}}{1-\text{spec}}$ — how much a positive result moves your belief

**The one idea, if you keep only one:** when a disease is rare, most positive tests are wrong, and no
amount of test quality fixes it. This is a fact about arithmetic, not about medicine — and it applies
just as forcefully to a machine learning model screening for a rare event as it does to a mammogram.

### Wednesday, September 9 — Studio

You will build the screening calculation yourself: simulate a population, count the four outcomes,
recover PPV empirically, and sweep prevalence to reproduce the curve from Step 5. It doubles as the
NumPy and pandas on-ramp, so no prior experience with either is assumed.